In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !unzip "/content/drive/MyDrive/archive.zip" -d "./medicinal_leaf/"

In [ ]:
import os
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# ImageDataGenerator for augmentation
datagen = ImageDataGenerator(
    rotation_range=30,          # Randomly rotate images by up to 30 degrees
    width_shift_range=0.2,      # Shift images horizontally by up to 20%
    height_shift_range=0.2,     # Shift images vertically by up to 20%
    shear_range=0.2,            # Shear transformation
    zoom_range=0.2,             # Zoom into images
    horizontal_flip=True,       # Randomly flip images horizontally
    fill_mode='nearest'         # Fill empty pixels after transformation
)

# Function to randomly select an image and generate augmented samples
image_list = ""
def augment_images_from_random_file(class_dir, num_samples=5):
    # List all files in the directory
    
    image_files = [f for f in image_list if f.endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    if not image_files:
        raise ValueError(f"No image files found in the directory: {class_dir}")
    
    # Randomly choose an image
    random_image = random.choice(image_files)
    
    # Load the image
    img_path = os.path.join(class_dir, random_image)
    img = load_img(img_path)
    x = img_to_array(img)
    x = x.reshape((1,) + x.shape)  # Reshape for the generator
    
    # Generate and save 'num_samples' augmented images
    for _ in range(num_samples):
        for batch in datagen.flow(x, batch_size=1, save_to_dir=class_dir, 
                                  save_prefix='aug', save_format='jpeg'):
            break  # We only need one batch per loop iteration
    
    print(f"Generated {num_samples} augmented images from {random_image}")

# Main augmentation loop
base_dir = './Medicinal_Leaves/'  # Root folder containing class subfolders
target_image_count = 500  # Target number of images per class

for class_name in os.listdir(base_dir):
    class_dir = os.path.join(base_dir, class_name)
    image_list = os.listdir(class_dir)
    # Count existing images in the class
    current_image_count = len([f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    
    if current_image_count < target_image_count:
        # Calculate how many images need to be generated
        images_needed = target_image_count - current_image_count
        
        while images_needed > 0:
            # Generate 5 augmented images from a randomly selected image
            augment_images_from_random_file(class_dir, num_samples=5)
            
            # Reduce the count of needed images
            images_needed -= 5
            
            # Adjust to avoid overshooting the target count
            if images_needed < 0:
                images_needed = 0

print("Augmentation completed!")
